# Capítulo 8 — PCA e Métodos Lineares: SVD, ICA e NMF

Códigos do capítulo. Execute as células em ordem; cada bloco é independente dos demais.


## 8.5 PCA no Python

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X, y = load_iris(return_X_y=True)
Xp = StandardScaler().fit_transform(X)

pca = PCA(n_components=4).fit(Xp)

print("Variância explicada   :", pca.explained_variance_ratio_.round(3))
print("Variância acumulada   :", pca.explained_variance_ratio_.cumsum().round(3))
print("Cargas (componentes):\n", pca.components_.round(2))

Z = PCA(n_components=2).fit_transform(Xp)
plt.scatter(Z[:, 0], Z[:, 1], c=y)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("Iris em 2 componentes")
plt.show()

## 8.6.1 Variância acumulada

In [ ]:
from sklearn.decomposition import PCA
# Pede ao PCA o nº de componentes suficiente para 95% da variância
pca95 = PCA(n_components=0.95)

## 8.8 PCA para reconstruir e comprimir

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

X, y = load_digits(return_X_y=True)          # 1797 imagens de 8x8 = 64 atributos
print("Original:", X.shape)

for k in [2, 5, 15, 40]:
    pca = PCA(n_components=k).fit(X)
    Xr = pca.inverse_transform(pca.transform(X))
    erro = np.mean((X - Xr) ** 2)
    var = pca.explained_variance_ratio_.sum()
    print(f"{k:3d} componentes -> variância {var:.2%}  erro quadrático médio {erro:.2f}")

## 8.10.1 PCA incremental

In [ ]:
import numpy as np
from sklearn.decomposition import IncrementalPCA

rng = np.random.default_rng(0)
ipca = IncrementalPCA(n_components=5, batch_size=500)
for _ in range(10):                      # simula 10 lotes lidos do disco
    lote = rng.normal(size=(500, 50))
    ipca.partial_fit(lote)
print(ipca.explained_variance_ratio_.round(3))

## 8.10.4 PCA com kernel

In [ ]:
import numpy as np
from sklearn.datasets import make_circles
from sklearn.decomposition import PCA, KernelPCA

X, y = make_circles(n_samples=400, factor=0.3, noise=0.03, random_state=0)
Z_lin = PCA(2).fit_transform(X)
Z_rbf = KernelPCA(n_components=2, kernel="rbf", gamma=10).fit_transform(X)

# No PCA linear os dois anéis continuam misturados na 1ª componente;
# no Kernel PCA a 1ª componente já separa os anéis.
lin_int, lin_ext = Z_lin[y == 1, 0].mean(), Z_lin[y == 0, 0].mean()
rbf_int, rbf_ext = Z_rbf[y == 1, 0].mean(), Z_rbf[y == 0, 0].mean()
print("PC1 linear: anel interno", round(lin_int, 2), "| externo", round(lin_ext, 2))
print("PC1 kernel: anel interno", round(rbf_int, 2), "| externo", round(rbf_ext, 2))

## 8.11.2 A ideia

In [ ]:
import numpy as np
from sklearn.decomposition import FastICA, PCA

t = np.linspace(0, 8, 2000)
s1 = np.sin(2 * t)                       # fonte 1: senoide
s2 = np.sign(np.sin(3 * t))              # fonte 2: onda quadrada
S = np.c_[s1, s2] + 0.05 * np.random.default_rng(0).normal(size=(2000, 2))

A = np.array([[1.0, 0.5], [0.4, 1.0]])   # mistura desconhecida
X = S @ A.T                              # sinais "gravados"

S_ica = FastICA(n_components=2, random_state=0).fit_transform(X)
S_pca = PCA(n_components=2).fit_transform(X)

# Correlação absoluta entre cada fonte original e cada componente recuperada
def melhor(S, R):
    c = np.abs(np.corrcoef(S.T, R.T)[:2, 2:])
    return c.max(axis=1).round(2)

print("ICA recupera as fontes? ", melhor(S, S_ica))
print("PCA recupera as fontes? ", melhor(S, S_pca))

## 8.12.2 Aplicação em texto: descoberta de tópicos

In [ ]:
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer

docs = [
    "o time venceu o jogo com gol no final",
    "o campeonato de futebol tem jogo decisivo",
    "o banco aumentou a taxa de juros",
    "a inflação e os juros preocupam o mercado",
    "o atacante marcou gol e o time venceu",
    "o mercado reage ao aumento dos juros e da inflação",
]
vet = TfidfVectorizer()
X = vet.fit_transform(docs)

nmf = NMF(n_components=2, init="nndsvda", random_state=0, max_iter=500)
W = nmf.fit_transform(X)
H = nmf.components_

palavras = vet.get_feature_names_out()
for i, topico in enumerate(H):
    top = topico.argsort()[::-1][:5]
    print(f"Tópico {i}:", [palavras[j] for j in top])

## Resumo do capítulo